# ModelShield — Benchmark Analysis Notebook

**Live 100-ticket AWS Bedrock benchmark comparison** between `modelshield-v1` (100% Amazon Nova Pro) and `modelshield-v2` (DynamoDB TTL cache + confidence-gated Nova Lite → Pro adaptive router).

Data sourced from `eval/results/benchmark_report.json` — measured live in `ap-southeast-2` with concurrency `2` and pacing `2.5 s`.

In [ ]:
import json
import os
from pathlib import Path

# Load benchmark report
REPORT_PATH = Path(os.path.dirname(os.path.abspath("__file__"))) / "results" / "benchmark_report.json"
if not REPORT_PATH.exists():
    REPORT_PATH = Path("eval/results/benchmark_report.json")

with open(REPORT_PATH) as f:
    report = json.load(f)

v1 = report["stacks"]["v1"]
v2 = report["stacks"]["v2"]
dataset = report["dataset"]

print(f"Benchmark mode:    {report['mode']}")
print(f"Generated at:      {report['generated_at']}")
print(f"Tickets:           {dataset['tickets_used']}")
print(f"Concurrency:       {report['concurrency']}")
print(f"Pacing:            {report['pacing_seconds']}s")

## 1. Dataset Distribution

The benchmark uses 100 labelled tickets with ground-truth severity (`P1`–`P4`) and category.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f1117'
matplotlib.rcParams['axes.facecolor'] = '#0f1117'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'

CYAN = '#06b6d4'
PURPLE = '#7c5cfc'
GREEN = '#22c55e'
AMBER = '#f59e0b'
ROSE = '#f43f5e'
PALETTE = [CYAN, PURPLE, GREEN, AMBER, ROSE, '#3b82f6']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Severity distribution
sev = dataset['by_severity']
ax1.bar(sev.keys(), sev.values(), color=[ROSE, AMBER, CYAN, PURPLE])
ax1.set_title('Tickets by Severity', fontweight='bold', color='white')
ax1.set_ylabel('Count')
for i, (k, v) in enumerate(sev.items()):
    ax1.text(i, v + 0.5, str(v), ha='center', fontweight='bold', color='white')

# Category distribution
cat = dataset['by_category']
ax2.barh(list(cat.keys()), list(cat.values()), color=PALETTE[:len(cat)])
ax2.set_title('Tickets by Category', fontweight='bold', color='white')
ax2.set_xlabel('Count')
for i, (k, v) in enumerate(cat.items()):
    ax2.text(v + 0.3, i, str(v), va='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 2. Head-to-Head Metrics Summary

Side-by-side comparison of `v1-before` (baseline) vs `v2-after` (adaptive router).

In [ ]:
m1 = v1['metrics']
m2 = v2['metrics']

metrics_table = [
    ('Requests',              m1['requests'],                    m2['requests'],                    ''),
    ('Successful',            m1['successful_requests'],         m2['successful_requests'],         ''),
    ('Accuracy',              f"{m1['accuracy']:.1%}",           f"{m2['accuracy']:.1%}",           f"{(m2['accuracy'] - m1['accuracy'])*100:+.2f} pp"),
    ('P1 Recall',             f"{m1['p1_recall']:.0%}",         f"{m2['p1_recall']:.0%}",          'Preserved'),
    ('P1 Precision',          f"{m1['p1_precision']:.0%}",      f"{m2['p1_precision']:.0%}",       ''),
    ('p95 Latency (ms)',      f"{m1['p95_latency_ms']:.1f}",    f"{m2['p95_latency_ms']:.1f}",     f"{(m2['p95_latency_ms']/m1['p95_latency_ms'] - 1)*100:+.1f}%"),
    ('Cost / 10K Tickets',    f"${m1['cost_per_10k_tickets_usd']:.4f}", f"${m2['cost_per_10k_tickets_usd']:.4f}", f"{(m2['cost_per_10k_tickets_usd']/m1['cost_per_10k_tickets_usd'] - 1)*100:+.1f}%"),
    ('Frontier Tokens',       f"{m1['total_frontier_tokens']:,}",f"{m2['total_frontier_tokens']:,}",f"{(m2['total_frontier_tokens']/m1['total_frontier_tokens'] - 1)*100:+.1f}%"),
    ('Lambda GB-Seconds',     f"{m1['total_lambda_gb_seconds']:.4f}", f"{m2['total_lambda_gb_seconds']:.4f}", f"{(m2['total_lambda_gb_seconds']/m1['total_lambda_gb_seconds'] - 1)*100:+.1f}%"),
    ('Cache Hits',            m1['cache_hits'],                  m2['cache_hits'],                  f"+{m2['cache_hits']}"),
    ('Escalations',           m1['escalations'],                 m2['escalations'],                 ''),
    ('Fallbacks',             m1['fallbacks'],                   m2['fallbacks'],                   ''),
]

print(f"{'Metric':<25} {'v1-before':>15} {'v2-after':>15} {'Improvement':>15}")
print('─' * 72)
for name, v1_val, v2_val, imp in metrics_table:
    print(f"{name:<25} {str(v1_val):>15} {str(v2_val):>15} {imp:>15}")

## 3. Cost Reduction Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Cost per 10K tickets
vals = [m1['cost_per_10k_tickets_usd'], m2['cost_per_10k_tickets_usd']]
bars = axes[0].bar(['v1 (Baseline)', 'v2 (Adaptive)'], vals, color=[AMBER, GREEN], width=0.5)
axes[0].set_title('Cost per 10K Tickets (USD)', fontweight='bold')
axes[0].set_ylabel('USD')
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'${val:.2f}', ha='center', fontweight='bold', color='white')

# Frontier tokens
vals = [m1['total_frontier_tokens'], m2['total_frontier_tokens']]
bars = axes[1].bar(['v1 (Baseline)', 'v2 (Adaptive)'], vals, color=[AMBER, GREEN], width=0.5)
axes[1].set_title('Frontier Tokens Used', fontweight='bold')
axes[1].set_ylabel('Tokens')
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000, f'{val:,}', ha='center', fontweight='bold', color='white')

# Lambda GB-seconds
vals = [m1['total_lambda_gb_seconds'], m2['total_lambda_gb_seconds']]
bars = axes[2].bar(['v1 (Baseline)', 'v2 (Adaptive)'], vals, color=[AMBER, GREEN], width=0.5)
axes[2].set_title('Lambda GB-Seconds', fontweight='bold')
axes[2].set_ylabel('GB-s')
for bar, val in zip(bars, vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.2f}', ha='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 4. Route Decision Distribution (`v2`)

How the adaptive router distributed tickets across cache hits, light-only, escalated, and direct-heavy routes.

In [ ]:
routes = m2.get('route_decisions', {})

labels = list(routes.keys())
values = list(routes.values())
colors = PALETTE[:len(labels)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
wedges, texts, autotexts = ax1.pie(values, labels=labels, autopct='%1.0f%%', colors=colors, textprops={'color': 'white'})
ax1.set_title('v2 Route Decisions', fontweight='bold')

# Bar chart
ax2.barh(labels, values, color=colors)
ax2.set_title('v2 Route Decision Counts', fontweight='bold')
ax2.set_xlabel('Tickets')
for i, v in enumerate(values):
    ax2.text(v + 0.5, i, str(v), va='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

print(f"\nv1 routes all 100 tickets via: {list(m1.get('route_decisions', {}).keys())}")
print(f"v2 cache hits: {m2['cache_hits']} | escalations: {m2['escalations']} | fallbacks: {m2['fallbacks']}")

## 5. Per-Ticket Latency Comparison

In [ ]:
v1_latencies = [r['telemetry']['latency_ms'] for r in v1['records'] if r.get('telemetry')]
v2_latencies = [r['telemetry']['latency_ms'] for r in v2['records'] if r.get('telemetry')]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Scatter: per-ticket latency
ax1.scatter(range(len(v1_latencies)), v1_latencies, alpha=0.5, s=12, color=AMBER, label='v1 (Nova Pro only)')
ax1.scatter(range(len(v2_latencies)), v2_latencies, alpha=0.5, s=12, color=CYAN, label='v2 (Adaptive)')
ax1.set_xlabel('Ticket Index')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Per-Ticket Latency', fontweight='bold')
ax1.legend(facecolor='#1a1a2e', edgecolor='white', labelcolor='white')

# Histogram overlay
ax2.hist(v1_latencies, bins=25, alpha=0.6, color=AMBER, label='v1')
ax2.hist(v2_latencies, bins=25, alpha=0.6, color=CYAN, label='v2')
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('Count')
ax2.set_title('Latency Distribution', fontweight='bold')
ax2.axvline(m1['p95_latency_ms'], color=AMBER, linestyle='--', alpha=0.8, label=f'v1 p95: {m1["p95_latency_ms"]:.0f}ms')
ax2.axvline(m2['p95_latency_ms'], color=CYAN, linestyle='--', alpha=0.8, label=f'v2 p95: {m2["p95_latency_ms"]:.0f}ms')
ax2.legend(facecolor='#1a1a2e', edgecolor='white', labelcolor='white')

plt.tight_layout()
plt.show()

## 6. Per-Ticket Cost Comparison

In [ ]:
v1_costs = [r['telemetry']['estimated_cost_usd'] for r in v1['records'] if r.get('telemetry')]
v2_costs = [r['telemetry']['estimated_cost_usd'] for r in v2['records'] if r.get('telemetry')]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Per-ticket cost scatter
ax1.scatter(range(len(v1_costs)), [c * 1000 for c in v1_costs], alpha=0.5, s=12, color=AMBER, label='v1')
ax1.scatter(range(len(v2_costs)), [c * 1000 for c in v2_costs], alpha=0.5, s=12, color=CYAN, label='v2')
ax1.set_xlabel('Ticket Index')
ax1.set_ylabel('Cost (mUSD)')
ax1.set_title('Per-Ticket Inference Cost', fontweight='bold')
ax1.legend(facecolor='#1a1a2e', edgecolor='white', labelcolor='white')

# Cumulative cost
v1_cum = [sum(v1_costs[:i+1]) for i in range(len(v1_costs))]
v2_cum = [sum(v2_costs[:i+1]) for i in range(len(v2_costs))]
ax2.plot(v1_cum, color=AMBER, linewidth=2, label=f'v1 total: ${v1_cum[-1]:.4f}')
ax2.plot(v2_cum, color=CYAN, linewidth=2, label=f'v2 total: ${v2_cum[-1]:.4f}')
ax2.fill_between(range(len(v1_cum)), v2_cum, v1_cum, alpha=0.15, color=GREEN)
ax2.set_xlabel('Ticket Index')
ax2.set_ylabel('Cumulative Cost (USD)')
ax2.set_title('Cumulative Inference Cost', fontweight='bold')
ax2.legend(facecolor='#1a1a2e', edgecolor='white', labelcolor='white')

savings = m1['total_cost_usd'] - m2['total_cost_usd']
pct = savings / m1['total_cost_usd'] * 100
print(f"\nTotal savings: ${savings:.4f} ({pct:.1f}% reduction)")

plt.tight_layout()
plt.show()

## 7. P1 Recall & Safety Verification

Critical check: every ground-truth `P1` incident must be correctly classified as `P1` by both stacks.

In [ ]:
print(f"{'Ticket ID':<12} {'Ground Truth':>14} {'v1 Prediction':>15} {'v2 Prediction':>15} {'v1 ✓':>6} {'v2 ✓':>6}")
print('─' * 72)

v2_map = {r['ticket_id']: r for r in v2['records']}

p1_v1_correct = 0
p1_v2_correct = 0
p1_total = 0

for r1 in v1['records']:
    gt = r1['ground_truth']['severity']
    if gt != 'P1':
        continue
    p1_total += 1
    r2 = v2_map.get(r1['ticket_id'], {})
    p1_v1 = r1.get('prediction', {}).get('severity', '?')
    p1_v2 = r2.get('prediction', {}).get('severity', '?')
    ok1 = '✅' if p1_v1 == 'P1' else '❌'
    ok2 = '✅' if p1_v2 == 'P1' else '❌'
    if p1_v1 == 'P1': p1_v1_correct += 1
    if p1_v2 == 'P1': p1_v2_correct += 1
    print(f"{r1['ticket_id']:<12} {'P1':>14} {p1_v1:>15} {p1_v2:>15} {ok1:>6} {ok2:>6}")

print(f"\nP1 Recall — v1: {p1_v1_correct}/{p1_total} ({p1_v1_correct/p1_total:.0%}) | v2: {p1_v2_correct}/{p1_total} ({p1_v2_correct/p1_total:.0%})")
print("🛡️  100% P1 recall preserved across both stacks." if p1_v1_correct == p1_total and p1_v2_correct == p1_total else "⚠️  P1 recall gap detected!")

## 8. Cache Hit Analysis (`v2`)

Tickets that hit the DynamoDB TTL cache cost `$0.00` in Bedrock inference.

In [ ]:
cache_hits = [r for r in v2['records'] if r.get('telemetry', {}).get('cache_hit')]
non_cache = [r for r in v2['records'] if not r.get('telemetry', {}).get('cache_hit')]

print(f"Cache hits: {len(cache_hits)} / {len(v2['records'])} tickets")
print(f"Inference cost saved by cache: ${sum(r['telemetry']['estimated_cost_usd'] for r in cache_hits):.6f}")

if cache_hits:
    print(f"\nCached ticket IDs:")
    for r in cache_hits:
        t = r['telemetry']
        print(f"  {r['ticket_id']:12s}  route={t['route_decision']:18s}  cost=${t['estimated_cost_usd']:.6f}  latency={t['latency_ms']}ms")
else:
    print("(No cache hits in this run — cache hits appear in repeat/burst ticket scenarios)")

## 9. Accuracy Confusion: Misclassified Tickets

In [ ]:
print("=== v1 Misclassified ===")
v1_wrong = [r for r in v1['records'] if not r.get('correct', True)]
for r in v1_wrong:
    gt = r['ground_truth']
    pr = r.get('prediction', {})
    print(f"  {r['ticket_id']:12s}  GT={gt['severity']}/{gt['category']:12s}  Pred={pr.get('severity','?')}/{pr.get('category','?'):12s}  conf={pr.get('confidence', 0):.2f}")
print(f"  Total: {len(v1_wrong)} / {m1['requests']}\n")

print("=== v2 Misclassified ===")
v2_wrong = [r for r in v2['records'] if not r.get('correct', True)]
for r in v2_wrong:
    gt = r['ground_truth']
    pr = r.get('prediction', {})
    t = r.get('telemetry', {})
    print(f"  {r['ticket_id']:12s}  GT={gt['severity']}/{gt['category']:12s}  Pred={pr.get('severity','?')}/{pr.get('category','?'):12s}  conf={pr.get('confidence', 0):.2f}  route={t.get('route_decision', '?')}")
print(f"  Total: {len(v2_wrong)} / {m2['requests']}")

## 10. Summary

| Metric | v1-before | v2-after | Δ |
| --- | --- | --- | --- |
| **P1 Recall** | 100% (11/11) | 100% (11/11) | Preserved |
| **Accuracy** | 94.0% | 92.0% | -2.0 pp |
| **Cost / 10K Tickets** | \$18.67 | \$3.30 | **-82.3%** |
| **Frontier Tokens** | 209,426 | 25,202 | **-88.0%** |
| **Lambda GB-Seconds** | 36.76 | 13.67 | **-62.8%** |
| **p95 Latency** | 929 ms | 815 ms | **-12.3%** |
| **Cache Hits** | 0 | 15 | +15 |

✅ The adaptive router (`v2`) delivers significant cost and compute savings while preserving 100% `P1` critical incident recall.